# Distance-Matched Control Analysis

Replicates the **connected vs unconnected** Mann-Whitney test from `compare_structural_functional.ipynb`, but with a critical improvement: each connected pair is matched to the **closest unconnected pair by 3D Euclidean distance** (`dist_um`, derived from `pt_position_x/y/z`).

Motivation: connected neurons tend to be spatially closer, so any elevated functional correlation in connected pairs could be driven by proximity rather than synaptic connectivity. Distance-matched controls isolate the connectivity effect.

Steps:
1. Build the same all-pairs table as in the main notebook
2. Greedily match each connected pair to its nearest (by `dist_um`) unconnected pair, without replacement
3. Re-run Mann-Whitney on the matched sample and compare to the unmatched result

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Libraries loaded.')

---
## 1. Load Data

In [ ]:
# Functional
F           = np.load('outputs/functional_network/F_correlation_matrix.npy')
func_cohort = pd.read_csv('outputs/functional_network/functional_cohort.csv')

# Structural — 93-neuron session 9.3 V1
s_nodes = pd.read_csv('data/exports/G_93_nodes.csv')
s_edges = pd.read_csv('data/exports/G_93_edges.csv')

print(f'Functional matrix shape : {F.shape}')
print(f'Structural nodes (93)   : {s_nodes.shape}')
print(f'Structural edges (93)   : {s_edges.shape}')

---
## 2. Merge and Build All-Pairs Table

In [ ]:
merged = func_cohort.merge(
    s_nodes[['neuron_id', 'pref_ori', 'gOSI',
             'out_degree', 'in_degree', 'out_strength', 'in_strength']],
    left_on='pt_root_id', right_on='neuron_id',
    how='inner'
).reset_index(drop=True)

assert len(merged) == 93, 'Expected 93 neurons after merge!'
id2midx = dict(zip(merged['pt_root_id'], merged['matrix_index']))

s_edges['pre_neuron_id']  = s_edges['pre_neuron_id'].astype('int64')
s_edges['post_neuron_id'] = s_edges['post_neuron_id'].astype('int64')
conn_set = set(zip(s_edges['pre_neuron_id'], s_edges['post_neuron_id']))

neurons = merged['pt_root_id'].values.astype('int64')
midxs   = merged['matrix_index'].values
xs = merged['pt_position_x'].values
ys = merged['pt_position_y'].values
zs = merged['pt_position_z'].values

rows = []
n = len(neurons)
for i in range(n):
    for j in range(i + 1, n):
        ni, nj = int(neurons[i]), int(neurons[j])
        fij = float(F[midxs[i], midxs[j]])
        c_ij = (ni, nj) in conn_set
        c_ji = (nj, ni) in conn_set
        if c_ij and c_ji:
            ctype = 'bidirectional'
        elif c_ij or c_ji:
            ctype = 'unidirectional'
        else:
            ctype = 'none'
        dist = np.sqrt((xs[i]-xs[j])**2 + (ys[i]-ys[j])**2 + (zs[i]-zs[j])**2)
        rows.append({'ni': ni, 'nj': nj,
                     'f_corr': fij,
                     'conn_type': ctype,
                     'connected': ctype != 'none',
                     'dist_um': dist})

pairs = pd.DataFrame(rows)
print(f'Total pairs:       {len(pairs):,}')
print(f'Connected pairs:   {pairs["connected"].sum():,}')
print(f'Unconnected pairs: {(~pairs["connected"]).sum():,}')
print(f'\nMean distance — connected:   {pairs[pairs["connected"]]["dist_um"].mean():.1f} µm')
print(f'Mean distance — unconnected: {pairs[~pairs["connected"]]["dist_um"].mean():.1f} µm')

---
## 3. Distance-Matched Sampling

For each connected pair (sorted by `dist_um` to reduce systematic bias), find the nearest-distance unconnected pair that hasn't been used yet. This is greedy 1:1 nearest-neighbour matching without replacement on `dist_um`.

In [ ]:
from scipy.spatial import KDTree

connected_df   = pairs[pairs['connected']].copy().reset_index(drop=True)
unconnected_df = pairs[~pairs['connected']].copy().reset_index(drop=True)

# Build a 1-D KD-tree on unconnected pair distances for fast nearest-neighbour lookup
unconn_dists = unconnected_df['dist_um'].values.reshape(-1, 1)
tree = KDTree(unconn_dists)

used = set()          # indices into unconnected_df already matched
matched_unconn_idx = []

# Process connected pairs ordered by their distance to minimise total matching error
conn_order = connected_df['dist_um'].argsort().values

for ci in conn_order:
    query = connected_df.loc[ci, 'dist_um']
    # Query enough neighbours to find one that hasn't been used yet
    k = min(len(unconnected_df), max(50, len(used) + 1))
    dists_nn, idxs_nn = tree.query([[query]], k=k)
    chosen = None
    for idx in idxs_nn[0]:
        if idx not in used:
            chosen = idx
            break
    if chosen is None:
        raise RuntimeError('Ran out of unconnected pairs to match against — '  
                           'this should not happen given 4 057 unconnected pairs.')
    used.add(chosen)
    matched_unconn_idx.append(chosen)

# Re-align back to original conn_order so rows correspond
# matched_unconn_idx[i] is the match for connected pair conn_order[i]
# Build the final matched frames in the original connected_df row order
match_map = dict(zip(conn_order, matched_unconn_idx))  # ci -> unconn idx
matched_unconn_rows = [match_map[ci] for ci in range(len(connected_df))]

matched_conn  = connected_df.copy()
matched_unconn = unconnected_df.loc[matched_unconn_rows].reset_index(drop=True)

# Distance residuals after matching
dist_resid = np.abs(matched_conn['dist_um'].values - matched_unconn['dist_um'].values)

print(f'Connected pairs matched:           {len(matched_conn):,}')
print(f'Unique unconnected controls used:  {len(used):,}')
print(f'\nMatching quality (|dist_conn - dist_control|):')
print(f'  mean  = {dist_resid.mean():.2f} µm')
print(f'  median= {np.median(dist_resid):.2f} µm')
print(f'  max   = {dist_resid.max():.2f} µm')
print(f'  95th  = {np.percentile(dist_resid, 95):.2f} µm')

---
## 4. Verify Distance Balance After Matching

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Before matching: distance distributions
ax = axes[0]
bins = np.linspace(0, pairs['dist_um'].max() * 1.02, 40)
ax.hist(pairs[~pairs['connected']]['dist_um'], bins=bins, density=True,
        alpha=0.55, color='#aab7c4', edgecolor='white', label='All unconnected')
ax.hist(pairs[pairs['connected']]['dist_um'], bins=bins, density=True,
        alpha=0.75, color='#E15759', edgecolor='white', label='Connected')
stat_b, p_b = stats.mannwhitneyu(
    pairs[pairs['connected']]['dist_um'],
    pairs[~pairs['connected']]['dist_um'])
ax.set_xlabel('3D distance (µm)')
ax.set_ylabel('Density')
ax.set_title(f'Before matching\nMW p={p_b:.4f}')
ax.legend(fontsize=8)

# After matching: distance distributions
ax2 = axes[1]
ax2.hist(matched_unconn['dist_um'], bins=bins, density=True,
         alpha=0.55, color='#aab7c4', edgecolor='white', label='Matched unconnected')
ax2.hist(matched_conn['dist_um'], bins=bins, density=True,
         alpha=0.75, color='#E15759', edgecolor='white', label='Connected')
stat_a, p_a = stats.mannwhitneyu(
    matched_conn['dist_um'], matched_unconn['dist_um'])
ax2.set_xlabel('3D distance (µm)')
ax2.set_ylabel('Density')
ax2.set_title(f'After matching\nMW p={p_a:.4f}  (want p≫0.05)')
ax2.legend(fontsize=8)

# Per-pair distance residuals
ax3 = axes[2]
ax3.hist(dist_resid, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax3.axvline(np.median(dist_resid), color='tomato', lw=2, ls='--',
            label=f'median={np.median(dist_resid):.1f} µm')
ax3.set_xlabel('|dist_connected − dist_control| (µm)')
ax3.set_ylabel('Count')
ax3.set_title('Per-pair distance residual after matching')
ax3.legend(fontsize=8)

plt.suptitle('Matching Quality: Distance Balance', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Before matching — mean dist connected: {pairs[pairs["connected"]]["dist_um"].mean():.1f} µm  '
      f'| unconnected: {pairs[~pairs["connected"]]["dist_um"].mean():.1f} µm')
print(f'After  matching — mean dist connected: {matched_conn["dist_um"].mean():.1f} µm  '
      f'| unconnected: {matched_unconn["dist_um"].mean():.1f} µm')

---
## 5. Mann-Whitney Test: Unmatched vs Distance-Matched

We run the same one-sided Mann-Whitney U test (connected > unconnected) twice:
- **Unmatched**: original 221 connected pairs vs all 4 057 unconnected pairs
- **Matched**: 221 connected pairs vs 221 distance-matched unconnected pairs

In [ ]:
conn_corr   = pairs[pairs['connected']]['f_corr'].values
unconn_corr = pairs[~pairs['connected']]['f_corr'].values
matched_conn_corr   = matched_conn['f_corr'].values
matched_unconn_corr = matched_unconn['f_corr'].values

# Unmatched test (reproduces Analysis 2 from main notebook)
stat_unmatched, p_unmatched = stats.mannwhitneyu(
    conn_corr, unconn_corr, alternative='greater')

# Matched test
stat_matched, p_matched = stats.mannwhitneyu(
    matched_conn_corr, matched_unconn_corr, alternative='greater')

def cohen_d(a, b):
    na, nb = len(a), len(b)
    pool = np.sqrt(((na-1)*a.std()**2 + (nb-1)*b.std()**2) / (na+nb-2))
    return (a.mean() - b.mean()) / pool

d_unmatched = cohen_d(conn_corr, unconn_corr)
d_matched   = cohen_d(matched_conn_corr, matched_unconn_corr)

print('=== Mann-Whitney U: connected > unconnected (one-sided) ===')
print()
print(f'UNMATCHED  (n_conn={len(conn_corr)}, n_unconn={len(unconn_corr)})')
print(f'  connected mean:   {conn_corr.mean():.5f} ± {conn_corr.std():.5f}')
print(f'  unconnected mean: {unconn_corr.mean():.5f} ± {unconn_corr.std():.5f}')
print(f'  U={stat_unmatched:.0f},  p={p_unmatched:.4g},  Cohen d={d_unmatched:.3f}')
print()
print(f'MATCHED    (n_conn={len(matched_conn_corr)}, n_unconn={len(matched_unconn_corr)})')
print(f'  connected mean:   {matched_conn_corr.mean():.5f} ± {matched_conn_corr.std():.5f}')
print(f'  unconnected mean: {matched_unconn_corr.mean():.5f} ± {matched_unconn_corr.std():.5f}')
print(f'  U={stat_matched:.0f},  p={p_matched:.4g},  Cohen d={d_matched:.3f}')

---
## 6. Visualise: Matched vs Unmatched Comparison

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

colors = {'connected': '#E15759', 'unmatched_unconn': '#aab7c4', 'matched_unconn': '#4E79A7'}

# ── Row 0: violins ──────────────────────────────────────────────────────────
ax_v1 = fig.add_subplot(gs[0, 0])
data_v1 = [unconn_corr, conn_corr]
labels_v1 = [f'Unconnected\n(n={len(unconn_corr):,})',
              f'Connected\n(n={len(conn_corr):,})']
colors_v1 = [colors['unmatched_unconn'], colors['connected']]
parts = ax_v1.violinplot(data_v1, positions=[1, 2], showmedians=True, showextrema=False)
for pc, c in zip(parts['bodies'], colors_v1):
    pc.set_facecolor(c); pc.set_alpha(0.75)
parts['cmedians'].set_color('black')
for i, (d, c) in enumerate(zip(data_v1, colors_v1)):
    jit = np.random.RandomState(i).uniform(-0.12, 0.12, min(len(d), 300))
    sample = np.random.RandomState(i).choice(d, min(len(d), 300), replace=False)
    ax_v1.scatter(np.full(len(sample), i+1) + jit, sample,
                  alpha=0.2, s=6, color=c, zorder=3)
ax_v1.set_xticks([1, 2])
ax_v1.set_xticklabels(labels_v1, fontsize=9)
ax_v1.set_ylabel('Pearson correlation')
ax_v1.set_title(f'Unmatched\np={p_unmatched:.4g}, d={d_unmatched:.3f}')

ax_v2 = fig.add_subplot(gs[0, 1])
data_v2 = [matched_unconn_corr, matched_conn_corr]
labels_v2 = [f'Matched control\n(n={len(matched_unconn_corr):,})',
              f'Connected\n(n={len(matched_conn_corr):,})']
colors_v2 = [colors['matched_unconn'], colors['connected']]
parts2 = ax_v2.violinplot(data_v2, positions=[1, 2], showmedians=True, showextrema=False)
for pc, c in zip(parts2['bodies'], colors_v2):
    pc.set_facecolor(c); pc.set_alpha(0.75)
parts2['cmedians'].set_color('black')
for i, (d, c) in enumerate(zip(data_v2, colors_v2)):
    jit = np.random.RandomState(i+10).uniform(-0.12, 0.12, len(d))
    ax_v2.scatter(np.full(len(d), i+1) + jit, d,
                  alpha=0.35, s=8, color=c, zorder=3)
ax_v2.set_xticks([1, 2])
ax_v2.set_xticklabels(labels_v2, fontsize=9)
ax_v2.set_ylabel('Pearson correlation')
ax_v2.set_title(f'Distance-matched\np={p_matched:.4g}, d={d_matched:.3f}')

# ── Summary table ────────────────────────────────────────────────────────────
ax_tbl = fig.add_subplot(gs[0, 2])
ax_tbl.axis('off')
tbl_data = [
    ['Unmatched', f'{len(conn_corr)}', f'{len(unconn_corr):,}',
     f'{conn_corr.mean():.4f}', f'{unconn_corr.mean():.4f}',
     f'{p_unmatched:.4g}', f'{d_unmatched:.3f}'],
    ['Matched', f'{len(matched_conn_corr)}', f'{len(matched_unconn_corr)}',
     f'{matched_conn_corr.mean():.4f}', f'{matched_unconn_corr.mean():.4f}',
     f'{p_matched:.4g}', f'{d_matched:.3f}'],
]
tbl_cols = ['Analysis', 'n conn', 'n ctrl',
            'mean conn', 'mean ctrl', 'MW p', "Cohen's d"]
tbl = ax_tbl.table(cellText=tbl_data, colLabels=tbl_cols,
                    cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1.1, 2.2)
ax_tbl.set_title('Test results summary', pad=16)

# ── Row 1: Overlapping histograms ────────────────────────────────────────────
ax_h1 = fig.add_subplot(gs[1, 0])
bins_h = np.linspace(-0.06, 0.30, 45)
ax_h1.hist(unconn_corr, bins=bins_h, density=True, alpha=0.5,
           color=colors['unmatched_unconn'], edgecolor='white',
           label=f'Unconnected (mean={unconn_corr.mean():.4f})')
ax_h1.hist(conn_corr, bins=bins_h, density=True, alpha=0.7,
           color=colors['connected'], edgecolor='white',
           label=f'Connected   (mean={conn_corr.mean():.4f})')
ax_h1.set_xlabel('Pearson r')
ax_h1.set_ylabel('Density')
ax_h1.set_title('Unmatched distributions')
ax_h1.legend(fontsize=8)

ax_h2 = fig.add_subplot(gs[1, 1])
ax_h2.hist(matched_unconn_corr, bins=bins_h, density=True, alpha=0.5,
           color=colors['matched_unconn'], edgecolor='white',
           label=f'Matched ctrl (mean={matched_unconn_corr.mean():.4f})')
ax_h2.hist(matched_conn_corr, bins=bins_h, density=True, alpha=0.7,
           color=colors['connected'], edgecolor='white',
           label=f'Connected    (mean={matched_conn_corr.mean():.4f})')
ax_h2.set_xlabel('Pearson r')
ax_h2.set_ylabel('Density')
ax_h2.set_title('Matched distributions')
ax_h2.legend(fontsize=8)

# ── Effect size comparison bar ────────────────────────────────────────────────
ax_eff = fig.add_subplot(gs[1, 2])
bar_labels = ["Unmatched\n(vs all unconn)", "Distance-matched\n(vs nearest ctrl)"]
bar_vals   = [d_unmatched, d_matched]
bar_colors = ['#59A14F', '#F28E2B']
bars = ax_eff.bar(bar_labels, bar_vals, color=bar_colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, bar_vals):
    ax_eff.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'd={val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_eff.axhline(0, color='gray', lw=0.8)
ax_eff.set_ylabel("Cohen's d  (connected − control)")
ax_eff.set_title("Effect size: unmatched vs matched")
ax_eff.set_ylim(bottom=min(0, min(bar_vals)) - 0.05)

fig.suptitle('Distance-Matched Control Analysis\n'
             'Does synaptic connectivity predict functional correlation beyond proximity?',
             fontweight='bold', fontsize=13)
plt.show()

---
## 7. Breakdown by Connection Type (Unidirectional / Bidirectional)

In [ ]:
uni_conn  = pairs[pairs['conn_type'] == 'unidirectional'].copy().reset_index(drop=True)
bi_conn   = pairs[pairs['conn_type'] == 'bidirectional'].copy().reset_index(drop=True)
unconn_df = pairs[pairs['conn_type'] == 'none'].copy().reset_index(drop=True)

def match_controls(target_df, pool_df, seed=42):
    """Greedy nearest-distance matching without replacement."""
    pool_dists = pool_df['dist_um'].values.reshape(-1, 1)
    t = KDTree(pool_dists)
    used = set()
    matched_idx = []
    order = target_df['dist_um'].argsort().values
    for ci in order:
        q = target_df.loc[ci, 'dist_um']
        k = min(len(pool_df), max(100, len(used) + 1))
        _, idxs = t.query([[q]], k=k)
        for idx in idxs[0]:
            if idx not in used:
                used.add(idx)
                matched_idx.append(idx)
                break
    map_ = dict(zip(order, matched_idx))
    return pool_df.loc[[map_[i] for i in range(len(target_df))]].reset_index(drop=True)

# Ensure pool has enough unique rows; sample without replacement from the full unconnected pool
uni_ctrl = match_controls(uni_conn, unconn_df)
bi_ctrl  = match_controls(bi_conn,  unconn_df)

def mw_test(a, b):
    stat, p = stats.mannwhitneyu(a, b, alternative='greater')
    d = cohen_d(a, b)
    return stat, p, d

stat_um, p_um, d_um = mw_test(conn_corr, unconn_corr)        # original unmatched
stat_mm, p_mm, d_mm = mw_test(matched_conn_corr, matched_unconn_corr)  # all-conn matched

stat_ui_um, p_ui_um, d_ui_um = mw_test(uni_conn['f_corr'].values, unconn_corr)
stat_ui_m,  p_ui_m,  d_ui_m  = mw_test(uni_conn['f_corr'].values, uni_ctrl['f_corr'].values)

stat_bi_um, p_bi_um, d_bi_um = mw_test(bi_conn['f_corr'].values, unconn_corr)
stat_bi_m,  p_bi_m,  d_bi_m  = mw_test(bi_conn['f_corr'].values, bi_ctrl['f_corr'].values)

print('=== Mann-Whitney results by connection type ===')
print()
print(f'{"Comparison":<40} {"n_conn":>6}  {"n_ctrl":>7}  {"mean_conn":>10}  {"mean_ctrl":>10}  {"p":>10}  {"d":>7}')
print('-'*100)
rows_r = [
    ('All connected (unmatched)',    len(conn_corr), len(unconn_corr), conn_corr.mean(), unconn_corr.mean(), p_um, d_um),
    ('All connected (matched)',      len(matched_conn_corr), len(matched_unconn_corr), matched_conn_corr.mean(), matched_unconn_corr.mean(), p_mm, d_mm),
    ('Unidirectional (unmatched)',   len(uni_conn), len(unconn_corr), uni_conn['f_corr'].mean(), unconn_corr.mean(), p_ui_um, d_ui_um),
    ('Unidirectional (matched)',     len(uni_conn), len(uni_ctrl), uni_conn['f_corr'].mean(), uni_ctrl['f_corr'].mean(), p_ui_m, d_ui_m),
    ('Bidirectional (unmatched)',    len(bi_conn), len(unconn_corr), bi_conn['f_corr'].mean(), unconn_corr.mean(), p_bi_um, d_bi_um),
    ('Bidirectional (matched)',      len(bi_conn), len(bi_ctrl), bi_conn['f_corr'].mean(), bi_ctrl['f_corr'].mean(), p_bi_m, d_bi_m),
]
for row in rows_r:
    label, nc, nctrl, mc, mctrl, p, d = row
    print(f'{label:<40} {nc:>6}  {nctrl:>7}  {mc:>10.5f}  {mctrl:>10.5f}  {p:>10.4g}  {d:>7.3f}')

---
## 8. Summary

| Analysis | n connected | n control | mean(conn) | mean(ctrl) | MW p (one-sided) | Cohen's d |
|----------|-------------|-----------|------------|------------|------------------|-----------|
| All connected — **unmatched** | 221 | 4 057 | ? | ? | ? | ? |
| All connected — **distance-matched** | 221 | 221 | ? | ? | ? | ? |
| Unidirectional — unmatched | 213 | 4 057 | ? | ? | ? | ? |
| Unidirectional — matched | 213 | 213 | ? | ? | ? | ? |
| Bidirectional — unmatched | 8 | 4 057 | ? | ? | ? | ? |
| Bidirectional — matched | 8 | 8 | ? | ? | ? | ? |

*(Fill in after running — numbers printed by cell 5 and cell 7 above.)*

**Interpretation guide:**
- If the matched p-value is still significant → the connectivity effect **survives** distance control; synaptic connection adds information beyond proximity.
- If the matched p-value is not significant but the unmatched one is → the apparent effect was driven by **spatial proximity** rather than connectivity *per se*.
- A reduction in Cohen's d after matching quantifies how much of the original effect was a distance confound.